# One gene → graph queries (BioInsight 4.4)

Walk through **BRCA1** using the BioInsight REST API, optional direct **Neo4j** Cypher, and federated **literature** links.

**Prerequisites** (~15 min): clone the repo, run `docker compose up --build`, then open this notebook. See [GETTING_STARTED.md](../docs/GETTING_STARTED.md).

> Association scores are correlative, not causal. Demo data only — not clinical-grade.

In [ ]:
from __future__ import annotations

import os
from collections import Counter
from pathlib import Path

import httpx
import pandas as pd
from IPython.display import display

API_BASE = os.getenv("BIOINSIGHT_API_URL", "http://localhost:8000/api/v1").rstrip("/")
GENE_SYMBOL = os.getenv("BIOINSIGHT_GENE", "BRCA1")

client = httpx.Client(base_url=API_BASE, timeout=30.0)


def get(path: str, **params):
    r = client.get(path, params=params)
    r.raise_for_status()
    return r.json()


print(f"API: {API_BASE}")
print(f"Gene: {GENE_SYMBOL}")

## 0. Health and provenance

In [ ]:
health = get("/health")
meta = get("/meta")
stats = get("/stats")

print("health:", health)
print("data_version:", meta.get("data_version"))
print("disclaimer:", meta.get("disclaimer", "")[:120], "…")
print("stats:", stats)

## 1. Resolve symbol → Ensembl ID

In [ ]:
hits = get("/genes", q=GENE_SYMBOL)
if not hits:
    raise SystemExit(f"No gene found for {GENE_SYMBOL!r}. Is Neo4j seeded?")

gene = next((g for g in hits if g["symbol"].upper() == GENE_SYMBOL.upper()), hits[0])
gene_id = gene["id"]
print(f"Resolved {gene['symbol']} → {gene_id}")
gene

## 2. Gene detail and ranked diseases

In [ ]:
detail = get(f"/genes/{gene_id}")
diseases = get(f"/genes/{gene_id}/diseases")

print(detail.get("name"), "|", detail.get("symbol"))

df_diseases = pd.DataFrame(diseases["diseases"])
df_diseases.head(10)

## 3. Evidence breakdown (decomposed association facts)

In [ ]:
evidence = get(f"/genes/{gene_id}/evidence", limit=10)

rows = []
for bundle in evidence["evidence"]:
    for item in bundle["evidence"]:
        rows.append(
            {
                "disease": bundle["disease_name"],
                "assoc_score": bundle["score"],
                "evidence_type": item["evidence_type"],
                "source": item["source"],
                "score": item.get("score"),
                "study_id": item.get("study_id"),
            }
        )

df_evidence = pd.DataFrame(rows)
display(df_evidence.head(12))

by_type = Counter(df_evidence["evidence_type"])
print("Evidence types (top associations):", dict(by_type))

In [ ]:
try:
    import matplotlib.pyplot as plt

    counts = pd.Series(dict(by_type)).sort_values(ascending=True)
    counts.plot(kind="barh", title=f"{GENE_SYMBOL} evidence types (sample)")
    plt.xlabel("count")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib for charts: pip install matplotlib")

## 4. Optional — direct Neo4j Cypher (1-hop neighborhood)

Same graph the API queries. Requires `.env` Neo4j credentials (default `bolt://localhost:7687`).

In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

try:
    from dotenv import load_dotenv
    from neo4j import GraphDatabase

    load_dotenv(ROOT / ".env")

    uri = os.getenv("NEO4J_URI", "bolt://localhost:7687")
    user = os.getenv("NEO4J_USER", "neo4j")
    password = os.getenv("NEO4J_PASSWORD", "changeme")

    driver = GraphDatabase.driver(uri, auth=(user, password))
    cypher = """
    MATCH (g:Gene {id: $id})-[r:ASSOCIATED_WITH]->(d:Disease)
    RETURN g.symbol AS gene, d.name AS disease, r.score AS score
    ORDER BY r.score DESC
    LIMIT 8
    """
    with driver.session() as session:
        records = session.run(cypher, id=gene_id).data()
    driver.close()

    pd.DataFrame(records)
except Exception as exc:
    print(f"Neo4j skipped ({exc}). REST steps above are enough for the tutorial.")

## 5. Literature and federated pointers

- **Structured graph** (this repo): scores + evidence types from Open Targets bulk ingest.
- **Federated IDs**: Ensembl, Open Targets Platform, UniProt.
- **Literature rows**: `europepmc` evidence sources in the graph; follow up on the web or via [kg-rag-demo](https://github.com/LordKay-sudo/kg-rag-demo) for cited document chunks.

In [ ]:
links = get(f"/genes/{gene_id}/external-links")
for link in links["links"]:
    print(f"{link['label']:20} {link['url']}")

lit_rows = df_evidence[df_evidence["source"].str.contains("pmc|literature", case=False, na=False)]
if not lit_rows.empty:
    print("\nLiterature-typed evidence in graph:")
    display(lit_rows)

epmc = f"https://www.ebi.ac.uk/europepmc/search?query={GENE_SYMBOL}"
print(f"\nEurope PMC search: {epmc}")

## Next steps

- UI: http://localhost:8080 — search the same gene, open graph view and compare page.
- MCP: [embabel-mcp](https://github.com/LordKay-sudo/embabel-mcp) prompt `public-data-to-mcp-tutorial`.
- Export: `GET /export/gene-report?gene_id=...&format=tsv` for analyst handoff.
- Schema: [ONTOLOGY_SCHEMA.md](../docs/ONTOLOGY_SCHEMA.md) · Benchmarks: [BENCHMARKS.md](../docs/BENCHMARKS.md).